# Design Patterns

## Strategy

The code below computes an order's total according to its shipping mode, with the pattern's classic form: a base class and one subclass per mode. None of these classes keeps any state.

Rewrite it without classes:

- two functions `standard_shipping(weight)` and `express_shipping(weight)`, which return the costs;
- a function `total(price, weight, shipping)`, which receives one of these two functions.

For an order of 30 € and 2 kg, you must get the same outputs as the original code: `35.0` and `44.0`. Then add a `store_pickup` mode, with no costs, without changing `total`: this order's total must be `30.0`.

In [ ]:
class ShippingMode:
  def cost(self, weight: float) -> float:
    raise NotImplementedError


class StandardShipping(ShippingMode):
  def cost(self, weight: float) -> float:
    return 5.0


class ExpressShipping(ShippingMode):
  def cost(self, weight: float) -> float:
    return 10.0 + 2.0 * weight


class Order:
  def __init__(self, price: float, weight: float, mode: ShippingMode) -> None:
    self.price = price
    self.weight = weight
    self.mode = mode

  def total(self) -> float:
    return self.price + self.mode.cost(self.weight)


print(Order(30.0, 2.0, StandardShipping()).total())
print(Order(30.0, 2.0, ExpressShipping()).total())

In [ ]:
# Your code here

### Additional hints

A shipping function takes the weight and returns the costs, like the `cost` method. `total` calls the function it receives: `shipping(weight)`.

```python
def standard_shipping(weight: float) -> float:
  return ...


def express_shipping(weight: float) -> float:
  return ...


def total(price: float, weight: float, shipping) -> float:
  return ...


print(total(30.0, 2.0, standard_shipping))
print(total(30.0, 2.0, express_shipping))
```

### Solution

In [ ]:
from collections.abc import Callable


def standard_shipping(weight: float) -> float:
  return 5.0


def express_shipping(weight: float) -> float:
  return 10.0 + 2.0 * weight


def store_pickup(weight: float) -> float:
  return 0.0


def total(price: float, weight: float, shipping: Callable[[float], float]) -> float:
  return price + shipping(weight)


print(total(30.0, 2.0, standard_shipping))
print(total(30.0, 2.0, express_shipping))
print(total(30.0, 2.0, store_pickup))

The base class and its subclasses disappear: `Callable[[float], float]` says all that `total` expects from a strategy. A class becomes useful again when the strategy keeps state or groups several methods.

## Adapter

The shop confirms its orders with a `Payment` object, whose `pay(euros)` method returns `True` if the payment is accepted. It wants to use `ThirdPartyCheckout`, a library it can't modify and whose interface is different: `charge(cents, currency)` takes a whole number of cents and returns a dictionary.

Write a `CheckoutAdapter` class, which receives a `ThirdPartyCheckout` and follows the `Payment` protocol: its `pay` method converts the amount to cents, calls `charge` with the currency `"EUR"`, and returns `True` if the status is `"accepted"`.

With a checkout whose limit is `10_000`, confirm order 1 of 19.99 €, then order 2 of 150 €. You must get `Charging 1999 cents (EUR)` and `Order 1 confirmed`, then `Charging 15000 cents (EUR)` and `Order 2 refused`.

In [ ]:
from typing import Protocol


class Payment(Protocol):
  def pay(self, euros: float) -> bool: ...


def confirm_order(number: int, total: float, payment: Payment) -> None:
  if payment.pay(total):
    print(f"Order {number} confirmed")
  else:
    print(f"Order {number} refused")


class ThirdPartyCheckout:
  def __init__(self, limit: int) -> None:
    self._limit = limit

  def charge(self, cents: int, currency: str) -> dict[str, str]:
    print(f"Charging {cents} cents ({currency})")
    status = "accepted" if cents <= self._limit else "refused"
    return {"status": status}

In [ ]:
# Your code here

### Additional hints

Keep the checkout in an attribute. For the conversion, use `round(euros * 100)` and not `int(euros * 100)`: `19.99 * 100` is `1998.9999999999998`, which `int` truncates to `1998`.

```python
class CheckoutAdapter:
  def __init__(self, checkout: ThirdPartyCheckout) -> None:
    self._checkout = checkout

  def pay(self, euros: float) -> bool:
    response = self._checkout.charge(..., "EUR")
    return ...


checkout = ThirdPartyCheckout(10_000)
confirm_order(1, 19.99, CheckoutAdapter(checkout))
confirm_order(2, 150.0, CheckoutAdapter(checkout))
```

### Solution

In [ ]:
class CheckoutAdapter:
  def __init__(self, checkout: ThirdPartyCheckout) -> None:
    self._checkout = checkout

  def pay(self, euros: float) -> bool:
    response = self._checkout.charge(round(euros * 100), "EUR")
    return response["status"] == "accepted"


checkout = ThirdPartyCheckout(10_000)
confirm_order(1, 19.99, CheckoutAdapter(checkout))
confirm_order(2, 150.0, CheckoutAdapter(checkout))

`CheckoutAdapter` doesn't inherit from `Payment`: the protocol describes what `confirm_order` expects, and a type checker verifies that the adapter follows it. Neither `confirm_order` nor `ThirdPartyCheckout` changed.

## Observer

A shop wants to trigger several actions when an order is paid (send a receipt, update the stock…), without the payment code knowing about these actions.

Write an `EventBus` class with two methods:

- `subscribe(event, callback)`, which subscribes a function to an event;
- `publish(event, *args)`, which calls, with `args` and in subscription order, every function subscribed to this event.

Subscribe `send_receipt` and `update_stock` to the `"order_paid"` event, then publish it for order 42: both messages must be printed. Publishing an event without subscribers must do nothing.

**Useful documentation**

- [`collections.defaultdict`](https://docs.python.org/3/library/collections.html#collections.defaultdict)

In [ ]:
def send_receipt(order_id: int) -> None:
  print(f"Receipt sent for order {order_id}")


def update_stock(order_id: int) -> None:
  print(f"Stock updated for order {order_id}")

In [ ]:
# Your code here

### Additional hints

Keep the subscribers in a dictionary that maps each event to the list of its functions: `defaultdict(list)` creates the list on the first subscription. In `publish`, `.get(event, [])` adds no entry for an event without subscribers. You must get `Receipt sent for order 42`, then `Stock updated for order 42`.

```python
from collections import defaultdict


class EventBus:
  def __init__(self) -> None:
    self._subscribers = defaultdict(list)

  def subscribe(self, event, callback) -> None: ...

  def publish(self, event, *args) -> None:
    for callback in self._subscribers.get(event, []):
      ...


bus = EventBus()
bus.subscribe("order_paid", send_receipt)
bus.subscribe("order_paid", update_stock)
bus.publish("order_paid", 42)
```

### Solution

In [ ]:
from collections import defaultdict
from collections.abc import Callable


class EventBus:
  def __init__(self) -> None:
    self._subscribers: defaultdict[str, list[Callable[..., None]]] = defaultdict(list)

  def subscribe(self, event: str, callback: Callable[..., None]) -> None:
    self._subscribers[event].append(callback)

  def publish(self, event: str, *args: object) -> None:
    for callback in self._subscribers.get(event, []):
      callback(*args)


bus = EventBus()
bus.subscribe("order_paid", send_receipt)
bus.subscribe("order_paid", update_stock)
bus.publish("order_paid", 42)
bus.publish("order_cancelled", 42)  # No subscriber: nothing happens

The code that publishes the event depends on no subscriber: you add an action by subscribing a function, without changing the payment code. The decorators lab writes the same mechanism as a decorator, with `@subscribe("order_confirmed")`.

## Iterator

The shop's departments form a tree: each `Tree` has a value and children. The next exercises write several traversals of this tree, so that the code traversing it doesn't have to know its structure.

### A Generator

Add to the `Tree` class below an `__iter__` method, written as a generator, that traverses the tree depth first: a node's value, then those of its children, in order. Then print `list(departments)`: you must get `['shop', 'computers', 'keyboards', 'books']`.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Tree:
  value: str
  children: list["Tree"] = field(default_factory=list)

  # Your code here


departments = Tree("shop", [Tree("computers", [Tree("keyboards")]), Tree("books")])

#### Additional hints

`yield self.value` gives the node's value. `yield from child` gives all those of a child: it calls this child's `__iter__`, and that's where the recursion is.

```python
from collections.abc import Iterator


def __iter__(self) -> Iterator[str]:
  yield ...
  for child in self.children:
    yield from ...
```

#### Solution

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass, field


@dataclass
class Tree:
  value: str
  children: list["Tree"] = field(default_factory=list)

  def __iter__(self) -> Iterator[str]:
    yield self.value
    for child in self.children:
      yield from child


departments = Tree("shop", [Tree("computers", [Tree("keyboards")]), Tree("books")])
print(list(departments))

### An Explicit Stack

The recursive generator adds one level of calls per level of the tree: on a very deep tree, it exceeds Python's recursion limit, as the next cell shows.

Write a `DepthFirstTraversal` class, which does the same traversal without recursion:

- `__init__(self, tree)` creates a stack, a list holding the root;
- `__iter__` returns `self`;
- `__next__` raises `StopIteration` if the stack is empty. Otherwise, it removes the stack's last node, adds its children, and returns its value.

`list(DepthFirstTraversal(departments))` must give the same result as `list(departments)`, and `len(list(DepthFirstTraversal(deep)))` must be `3001`.

In [ ]:
deep = Tree("leaf")
for number in range(3000):
  deep = Tree(f"level {number}", [deep])

try:
  list(deep)
except RecursionError:
  print("RecursionError: too many levels for the recursive generator")

In [ ]:
# Your code here

#### Additional hints

The stack gives back the node added last first: add the children from last to first with `reversed(node.children)`, so that the first child comes out first.

```python
class DepthFirstTraversal:
  def __init__(self, tree: Tree) -> None:
    self._stack = [tree]

  def __iter__(self) -> "DepthFirstTraversal":
    return self

  def __next__(self) -> str:
    if not self._stack:
      raise ...
    node = self._stack.pop()
    self._stack.extend(...)
    return ...


print(list(DepthFirstTraversal(departments)))
print(len(list(DepthFirstTraversal(deep))))
```

#### Solution

In [ ]:
class DepthFirstTraversal:
  def __init__(self, tree: Tree) -> None:
    self._stack = [tree]

  def __iter__(self) -> "DepthFirstTraversal":
    return self

  def __next__(self) -> str:
    if not self._stack:
      raise StopIteration
    node = self._stack.pop()
    self._stack.extend(reversed(node.children))
    return node.value


print(list(DepthFirstTraversal(departments)))
print(len(list(DepthFirstTraversal(deep))))

The traversal's position is in the iterator's stack, not in `Tree` or in Python's call stack: this is the GoF's iterator, and two traversals of the same tree don't interfere.

### A Second Traversal (Optional)

Write a `BreadthFirstTraversal` class, which traverses the tree level by level, with a `collections.deque` queue instead of the stack. `list(BreadthFirstTraversal(departments))` must give `['shop', 'computers', 'books', 'keyboards']`.

**Useful documentation**

- [`collections.deque`](https://docs.python.org/3/library/collections.html#collections.deque)

In [ ]:
# Your code here

#### Additional hints

`popleft()` removes the oldest node of the queue, and `extend` adds the children at its end, in their order.

```python
from collections import deque


class BreadthFirstTraversal:
  def __init__(self, tree: Tree) -> None:
    self._queue = deque([tree])

  def __iter__(self) -> "BreadthFirstTraversal":
    return self

  def __next__(self) -> str:
    if not self._queue:
      raise StopIteration
    node = self._queue.popleft()
    self._queue.extend(...)
    return ...


print(list(BreadthFirstTraversal(departments)))
```

#### Solution

In [ ]:
from collections import deque


class BreadthFirstTraversal:
  def __init__(self, tree: Tree) -> None:
    self._queue = deque([tree])

  def __iter__(self) -> "BreadthFirstTraversal":
    return self

  def __next__(self) -> str:
    if not self._queue:
      raise StopIteration
    node = self._queue.popleft()
    self._queue.extend(node.children)
    return node.value


print(list(BreadthFirstTraversal(departments)))

`Tree` didn't change: the point of the pattern is to offer several traversals of one structure, without the code using them knowing that structure.